Goal: 
Preprocess the data with supervised learning technique

Approach: 
Combines entity recognition (NER) preprocessing with a supervised relation extraction model (BertRelationExtractor), trained on a small dataset of sentences, entities, and labeled relations.

In [1]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification, BertModel
from TorchCRF import CRF
from torch import nn
from torch.utils.data import DataLoader, Dataset
import numpy as np

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score, f1_score

In [2]:
label_map = {
    "ENTITY": 0, 
    "TIME": 1, 
    "UNIT": 2,
    "NUMBER": 3,
}

Assume the dataset is normalized beforehand

In [7]:
train_sentences = [
    "profit for the year was 24.0 million dollars , 372.0 million dollars and 1,096.4 million dollars in 2021 , 2022 and 2023 , respectively.",
    "adjusted profit was 769.6 million dollars , 788.1 million dollars and 1,459.0 million dollars in 2021 , 2022 and 2023 , respectively.",
    "The revenue in 2021 was 4.38 billion dollars",
    "The revenue in 2022 was 5.56 billion dollars",
    "The revenue in 2023 was 7.68 billion dollars",
    "Profit in 2021 was 24 million dollars",
    "Profit in 2022 was 372 million dollars",
    "Profit in 2023 was 1.10 billion dollars",
    "Adjusted profit in 2021 was 769.6 million dollars",
    "Adjusted profit in 2022 was 788.1 million dollars",
    "Adjusted profit in 2023 was 1.46 billion dollars"
]

train_entities = [
    [("profit", "ENTITY", 0), ("24.0 million dollars", "NUMBER", 5), ("372.0 million dollars", "NUMBER", 9), ("1,096.4 million dollars", "NUMBER", 13), ("2021", "TIME", 17), ("2022", "TIME", 19), ("2023", "TIME", 21)],
    [("adjusted profit", "ENTITY", 0), ("769.6 million dollars", "NUMBER", 3), ("788.1 million dollars", "NUMBER", 7), ("1,459.0 million dollars", "NUMBER", 11), ("2021", "TIME", 15), ("2022", "TIME", 17), ("2023", "TIME", 19)],
    [("revenue", "ENTITY", 1), ("2021", "TIME", 3), ("4.38 billion dollars", "NUMBER", 5)],
    [("revenue", "ENTITY", 1), ("2022", "TIME", 3), ("5.56 billion dollars", "NUMBER", 5)],
    [("revenue", "ENTITY", 1), ("2023", "TIME", 3), ("7.68 billion dollars", "NUMBER", 5)],
    [("Profit", "ENTITY", 0), ("2021", "TIME", 2), ("24 million dollars", "NUMBER", 4)],
    [("Profit", "ENTITY", 0), ("2022", "TIME", 2), ("372 million dollars", "NUMBER", 4)],
    [("Profit", "ENTITY", 0), ("2023", "TIME", 2), ("1.10 billion dollars", "NUMBER", 4)],
    [("Adjusted profit", "ENTITY", 0), ("2021", "TIME", 3), ("769.6 million dollars", "NUMBER", 5)],
    [("Adjusted profit", "ENTITY", 0), ("2022", "TIME", 3), ("788.1 million dollars", "NUMBER", 5)],
    [("Adjusted profit", "ENTITY", 0), ("2023", "TIME", 3), ("1.46 billion dollars", "NUMBER", 5)]
]

train_relations = [
    [(0, 5, "has_value"), (5, 17, "in_year"), (0, 9, "has_value"), (9, 19, "in_year"), (0, 13, "has_value"), (13, 21, "in_year")],
    [(0, 3, "has_value"), (3, 15, "in_year"), (0, 7, "has_value"), (7, 17, "in_year"), (0, 11, "has_value"), (11, 19, "in_year")],
    [(1, 5, "has_value"), (5, 3, "in_year")],
    [(1, 5, "has_value"), (5, 3, "in_year")],
    [(1, 5, "has_value"), (5, 3, "in_year")],
    [(0, 4, "has_value"), (4, 2, "in_year")],
    [(0, 4, "has_value"), (4, 2, "in_year")],
    [(0, 4, "has_value"), (4, 2, "in_year")],
    [(0, 5, "has_value"), (5, 3, "in_year")],
    [(0, 5, "has_value"), (5, 3, "in_year")],
    [(0, 5, "has_value"), (5, 3, "in_year")]
]


In [4]:
class BertRelationExtractor(nn.Module):
    def __init__(self, num_labels, dropout=0.1):
        super(BertRelationExtractor, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size * 2, num_labels)  # Concatenate two entity embeddings

    def forward(self, input_ids, attention_mask, entity1_mask, entity2_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        hidden_states = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]

        # Extract embeddings for entity1 and entity2 using their masks
        entity1_embeds = (hidden_states * entity1_mask.unsqueeze(-1)).sum(dim=1)  # Sum over entity1 tokens
        entity2_embeds = (hidden_states * entity2_mask.unsqueeze(-1)).sum(dim=1)  # Sum over entity2 tokens
        
        # Normalize by number of tokens (optional, to handle multi-token entities)
        entity1_embeds = entity1_embeds / entity1_mask.sum(dim=1, keepdim=True).clamp(min=1)
        entity2_embeds = entity2_embeds / entity2_mask.sum(dim=1, keepdim=True).clamp(min=1)

        # Concatenate embeddings
        combined_embeds = torch.cat([entity1_embeds, entity2_embeds], dim=-1)
        combined_embeds = self.dropout(combined_embeds)
        
        logits = self.classifier(combined_embeds)  # [batch_size, num_labels]

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
            return loss
        return logits

In [5]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
relation_map = {"no_relation": 0, "has_value": 1, "in_year": 2, "owned_by": 3}
max_length = 100

In [6]:
def preprocess_relations(sentences, entities, relations):
    inputs = {"input_ids": [], "attention_mask": [], "entity1_mask": [], "entity2_mask": [], "labels": []}
    for sent, ent_list, rel_list in zip(sentences, entities, relations):
        encoding = tokenizer(sent, return_tensors="pt", padding="max_length", max_length=max_length, truncation=True, return_offsets_mapping=True)
        input_ids = encoding["input_ids"][0]
        attention_mask = encoding["attention_mask"][0]
        offsets = encoding["offsets_mapping"][0]
        tokens = sent.split()
        for i, (e1_text, e1_label, e1_idx) in enumerate(ent_list):
            for j, (e2_text, e2_label, e2_idx) in enumerate(ent_list):
                if i == j:
                    continue
                e1_mask = torch.zeros(max_length)
                e2_mask = torch.zeros(max_length)
                e1_start_char = sum(len(t) + 1 for t in tokens[:e1_idx])
                e2_start_char = sum(len(t) + 1 for t in tokens[:e2_idx])
                for idx, (start, end) in enumerate(offsets):
                    if start >= e1_start_char and start < e1_start_char + len(e1_text):
                        e1_mask[idx] = 1
                    if start >= e2_start_char and start < e2_start_char + len(e2_text):
                        e2_mask[idx] = 1
                rel_label = next((relation_map[rel] for (idx1, idx2, rel) in rel_list if idx1 == e1_idx and idx2 == e2_idx), relation_map["no_relation"])
                inputs["input_ids"].append(input_ids)
                inputs["attention_mask"].append(attention_mask)
                inputs["entity1_mask"].append(e1_mask)
                inputs["entity2_mask"].append(e2_mask)
                inputs["labels"].append(torch.tensor(rel_label, dtype=torch.long))
    return {k: torch.stack(v) for k, v in inputs.items()}

In [20]:
train_data = preprocess_relations(train_sentences, train_entities, train_relations)

In [ ]:
def split_data(data, val_split=0.2, random_seed=42):
    # Number of samples
    num_samples = len(data["input_ids"])
    indices = list(range(num_samples))
    
    # Split indices
    train_indices, val_indices = train_test_split(indices, test_size=val_split, random_state=random_seed)
    
    # Create train and val dictionaries
    train_split = {}
    val_split = {}
    for key in data:
        train_split[key] = data[key][train_indices]
        val_split[key] = data[key][val_indices]
    
    return train_split, val_split

In [ ]:
train_split, val_split = split_data(train_data, val_split=0.2)

In [21]:
class RelationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data["input_ids"])

    def __getitem__(self, idx):
        return {key: self.data[key][idx] for key in self.data}

In [22]:
train_dataset = RelationDataset(train_split)
val_dataset = RelationDataset(val_split)

batch_size = 8
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [13]:
print({k: v.shape for k, v in train_data.items()})

{'input_ids': torch.Size([30, 100]), 'attention_mask': torch.Size([30, 100]), 'entity1_mask': torch.Size([30, 100]), 'entity2_mask': torch.Size([30, 100]), 'labels': torch.Size([30])}


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
relation_map = {"no_relation": 0, "has_value": 1, "in_year": 2, "owned_by": 3}

model = BertRelationExtractor(num_labels=len(relation_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
best_val_loss = float('inf')
patience_counter = 0
best_model_state = None

In [22]:
def train_model(model, train_loader, val_loader, epochs=5, patience=3):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    loss_fn = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            e1_mask = batch["entity1_mask"].to(device)
            e2_mask = batch["entity2_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            loss = model(input_ids, attention_mask, e1_mask, e2_mask, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)

        # Validation phase
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                e1_mask = batch["entity1_mask"].to(device)
                e2_mask = batch["entity2_mask"].to(device)
                labels = batch["labels"].to(device)

                logits = model(input_ids, attention_mask, e1_mask, e2_mask)
                loss = loss_fn(logits, labels)
                val_loss += loss.item()

                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')  # Weighted F1 for multi-class
        
        print(f"Epoch {epoch+1}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Val Accuracy: {val_accuracy:.4f}")
        print(f"  Val F1-Score: {val_f1:.4f}")

        #Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict()  # Save best model weights
            patience_counter = 0
            print(f"  Validation loss improved to {best_val_loss:.4f}, saving model state.")
        else:
            patience_counter += 1
            print(f"  Validation loss did not improve. Patience counter: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
    
    return model

In [23]:
train_model(model, train_loader, val_loader, epochs=5, patience=3)

Epoch 1, Loss: 1.4377
Epoch 2, Loss: 0.8303
Epoch 3, Loss: 0.7330
Epoch 4, Loss: 0.7583
Epoch 5, Loss: 0.7092
